In [ ]:
import subprocess, sys, traceback, os

CKPT = "/kaggle/working/checkpoint.txt"

def ckpt(msg):
    print(msg, flush=True)
    with open(CKPT, "a") as f:
        f.write(msg + "\n")

def run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(f"$ {' '.join(cmd)}\n{r.stdout[-3000:]}\n{r.stderr[-3000:]}", flush=True)
    r.check_returncode()
    return r

try:
    ckpt("STEP0: /kaggle/input listing")
    for root, dirs, files in os.walk("/kaggle/input"):
        ckpt(f"  {root}: dirs={dirs} files={files[:10]}")

    ckpt("STEP1: pip install tabicl")
    run([sys.executable, "-m", "pip", "install", "-q", "tabicl"])

    ckpt("STEP1: git clone")
    subprocess.run(["rm", "-rf", "kaggle_playground_s6e9"])
    run(["git", "clone", "-q", "https://github.com/gccarno/kaggle_playground_s6e9.git"])

    import torch
    ckpt(f"STEP1: cuda available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        ckpt(f"STEP1: device: {torch.cuda.get_device_name(0)}")
        ckpt(f"STEP1: mem GB: {torch.cuda.get_device_properties(0).total_memory / 1e9}")
    ckpt("STEP1: done")
except Exception:
    tb = traceback.format_exc()
    print(tb, flush=True)
    with open(CKPT, "a") as f:
        f.write("STEP1 FAILED\n" + tb)
    raise

In [ ]:
import json, os, sys, time, traceback

CKPT = "/kaggle/working/checkpoint.txt"

def ckpt(msg):
    print(msg, flush=True)
    with open(CKPT, "a") as f:
        f.write(msg + "\n")

# P1 is a SINGLE-FOLD diagnostic, not a full pipeline.py run: pipeline.py's standard
# loop predicts the full 286,571-row TEST SET once per outer fold (5x total), which is
# cheap for trees but would make TabICL at large context sizes take many hours for no
# reason -- the actual question (does more context beat the champion's per-fold AUC)
# only needs one fold's real, full-validation-set number. Deliberately does not touch
# test.csv or emit a submission.csv; this is architecture research, not a champion
# candidate to ship (KAGGLE_PLAYBOOK.md s7: representation over architecture).
try:
    ckpt("STEP2: importing pipeline.py")
    sys.path.insert(0, "kaggle_playground_s6e9/src")
    import numpy as np
    import pandas as pd
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score
    import pipeline as pl
    from tabicl import TabICLClassifier

    CFG = {
        **pl.DEFAULTS,
        "te_cols": ["Annual_Income_USD", "Daily_Commute_km", "Age"],
        "te_smooth": 5.0,
        "te_backoff": "neighborhood",
    }
    MAX_CONTEXT = 450000
    PREDICT_CHUNK = 5000
    N_ESTIMATORS = 4

    ckpt("STEP2: loading data + building features")
    train = pd.read_csv(pl.DATA_DIR / "train.csv")
    test = pd.read_csv(pl.DATA_DIR / "test.csv")
    y = (train[pl.TARGET] == "Yes").astype(int).values
    train, test, feats_num, feats_cat = pl.base_features(train, test, CFG)
    feats = feats_num + feats_cat

    skf = StratifiedKFold(CFG["n_folds"], shuffle=True, random_state=CFG["cv_seed"])
    i, j = next(iter(skf.split(train[feats], y)))
    Xtr, Xva = train[feats].iloc[i].copy(), train[feats].iloc[j].copy()
    Xte_unused = test[feats].copy()
    ytr, yva = y[i], y[j]

    ckpt("STEP2: fitting target encoders on fold 0")
    pl.apply_target_encoding(Xtr, ytr, Xva, Xte_unused, CFG["te_cols"], CFG, CFG["cv_seed"])
    ckpt(f"STEP2: features = {list(Xtr.columns)}")

    n = min(MAX_CONTEXT, len(Xtr))
    idx = np.random.default_rng(42).choice(len(Xtr), size=n, replace=False)
    Xs, ys = Xtr.iloc[idx], ytr[idx]
    ckpt(f"STEP2: fitting TabICL on {n} context rows (of {len(Xtr)} available)")

    t0 = time.time()
    clf = TabICLClassifier(random_state=42, n_estimators=N_ESTIMATORS)
    clf.fit(Xs, ys)
    ckpt(f"STEP2: fit done in {time.time() - t0:.1f}s")

    preds = np.zeros(len(Xva))
    for k in range(0, len(Xva), PREDICT_CHUNK):
        tb = time.time()
        preds[k:k + PREDICT_CHUNK] = clf.predict_proba(Xva.iloc[k:k + PREDICT_CHUNK])[:, 1]
        if k % (PREDICT_CHUNK * 10) == 0:
            ckpt(f"  predicted {k}:{k + PREDICT_CHUNK} in {time.time() - tb:.1f}s")

    auc = roc_auc_score(yva, preds)
    metrics = {
        "run_tag": "P1", "learner": "tabicl_diag", "final_oof_auc": round(float(auc), 6),
        "n_context": n, "n_val": len(Xva), "n_estimators": N_ESTIMATORS,
        "wall_sec": round(time.time() - t0, 1),
    }
    ckpt(f"STEP2: fold-0 AUC = {auc:.6f}  (context={n}, val={len(Xva)})")
    print("RUN_METRICS_JSON:" + json.dumps(metrics), flush=True)

    np.save("/kaggle/working/p1_fold0_proba.npy", preds)
    np.save("/kaggle/working/p1_fold0_idx.npy", j)
    ckpt("STEP2: done")
except Exception:
    tb = traceback.format_exc()
    print(tb, flush=True)
    with open(CKPT, "a") as f:
        f.write("STEP2 FAILED\n" + tb)
    raise